In [1]:
import sys
import os
import importlib.util

print(f"--- Current Working Directory (Folder 1) ---")
print(os.getcwd())

print("\n--- Python's Search Path (sys.path) ---")
for path in sys.path:
    print(path)

print("\n--- Where is Python finding 'focal_loss'? ---")
spec = importlib.util.find_spec("focal_loss")

if spec:
    print(f"Found 'focal_loss' package at: {spec.origin}")
    if os.getcwd() in spec.origin:
        print("\n*** PROBLEM FOUND: You are importing a local file, not the package! ***")
        print("Please rename any file named 'focal_loss.py' in your current directory.")
    else:
        print("\nThis looks correct. The package is being loaded from your venv.")
else:
    print("CRITICAL: Python cannot find 'focal_loss' at all.")
    print("This suggests a corrupted venv or sys.path issue.")

--- Current Working Directory (Folder 1) ---
/home/poorna/capstone/first_implementation

--- Python's Search Path (sys.path) ---
/usr/lib64/python311.zip
/usr/lib64/python3.11
/usr/lib64/python3.11/lib-dynload

/home/poorna/venvs/torch/lib64/python3.11/site-packages
/home/poorna/venvs/torch/lib/python3.11/site-packages

--- Where is Python finding 'focal_loss'? ---
Found 'focal_loss' package at: /home/poorna/venvs/torch/lib64/python3.11/site-packages/focal_loss/__init__.py

This looks correct. The package is being loaded from your venv.


In [2]:
!which python

/home/poorna/venvs/torch/bin/python


In [3]:
# ==================================================================================
# REFACTORED EEG-TO-TEXT MODEL TRAINING SCRIPT (V2 - WITH SCHEDULED SAMPLING)
# Compatible with NEW Qwen Dataset (color + 90 objects, NO categories)
# ==================================================================================
#
# --- V1 CHANGES (Refactoring for Qwen Dataset) ---
# 1.  Updated NUM_COLORS from 77 to 12 (matches new dataset)
# 2.  Removed NUM_CATEGORIES entirely (not in new dataset)
# 3.  Updated NUM_OBJECTS from 61 to 90 (matches new dataset)
# 4.  Modified MetadataEncoder to handle [color_id, 90_objects]
# 5.  Removed category_criterion and all category prediction logic
# 6.  Updated model forward pass to return only (text, color, objects)
# 7.  Updated training/eval loops to skip category loss
# 8.  Updated HDF5 file path to new dataset
# 9.  Fixed metadata tensor indexing throughout (no more meta[:, 1] for category)
#
# --- V2 CHANGES (Methodological Fixes for Mode Collapse) ---
# 10. Added high loss weights (COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT) to the
#     main training loop and passed them to train/evaluate functions.
# 11. Modified Seq2Seq.forward() to accept 'meta_teacher_forcing_ratio':
#     - This 'schedules' the use of metadata.
#     - At ratio=1.0, it uses 100% ground-truth metadata (old way).
#     - At ratio=0.5, it uses 50% ground-truth and 50% *predicted* metadata.
# 12. Added logic to Seq2Seq.forward() to create a predicted metadata vector
#     from the meta_head's output when not using the ground truth.
# 13. Modified train_one_epoch() to accept and pass 'meta_teacher_forcing_ratio'
#     to the model, forcing the meta_head and decoder to learn jointly.
# 14. Implemented a linear annealing schedule for 'meta_teacher_forcing_ratio'
#     in the main training loop (from 1.0 down to 0.5).
# 15. Updated train_one_epoch() and evaluate() to return all individual losses
#     (total, text, color, object) for better diagnostics.
# 16. Fixed evaluation metrics crash by importing 'evaluate as hf_evaluate'.
# ==================================================================================

import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [14]:
# ==================================================================================
# MODIFIED Focal Loss Setup (Forcing Fallback to BCEWithLogitsLoss)
#
# WHY:
# The 'focal-loss' package is for MULTI-CLASS problems (targets=[1, 4, 0])
# Our problem is MULTI-LABEL (targets=[0, 1, 1, 0]).
# Therefore, 'focal-loss' is incompatible.
#
# We are now forcing the use of nn.BCEWithLogitsLoss, which is the correct
# PyTorch loss function for multi-label classification.
# ==================================================================================
print("WARNING: Third-party focal loss packages are incompatible with this multi-label task.")
print("Falling back to nn.BCEWithLogitsLoss, which is the correct loss function.")
focal_loss_criterion = nn.BCEWithLogitsLoss()
# ==================================================================================
# UPDATED CONSTANTS TO MATCH NEW DATASET
# ==================================================================================
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"  # NEW DATASET PATH
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 16

# --- Updated metadata dimensions ---
NUM_COLORS = 12      # Changed from 77 (matches new color map)
# NUM_CATEGORIES = REMOVED (not present in new dataset)
NUM_OBJECTS = 90     # Changed from 61 (matches new object map)

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Metadata config: {NUM_COLORS} colors, {NUM_OBJECTS} objects (NO categories)")

# ==================================================================================
# GRANGER CAUSALITY (Unchanged)
# ==================================================================================
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            min_len = 20
            if len(ts_i) < min_len or len(ts_j) < min_len:
                causality_matrix[i, j] = 0.0
                continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1:
                    causality_matrix[i, j] = 0.0
                    continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except Exception as e:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)

    if edge_attr is None:
        edge_attr = torch.tensor([], dtype=torch.float)
    elif edge_attr.ndim == 0:
        edge_attr = edge_attr.unsqueeze(0)

    return edge_index.to(torch.long), edge_attr.to(torch.float)


Falling back to nn.BCEWithLogitsLoss, which is the correct loss function.
Using device: cuda
Metadata config: 12 colors, 90 objects (NO categories)


In [15]:
# ==================================================================================
# DATASET AND DATALOADER (Unchanged)
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))

        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

In [16]:
# ==================================================================================
# MODEL COMPONENTS
# ==================================================================================

# --- SpatioTemporalEEGEncoder (Unchanged) ---
class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)
        print(f"Encoder RNN input size: {enc_hidden}")

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden

# --- LuongAttention (Unchanged) ---
class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)


In [17]:

# ==================================================================================
# REFACTORED METADATA ENCODER - MAJOR CHANGE
# ==================================================================================
class MetadataEncoder(nn.Module):
    """
    REFACTORED to handle new metadata structure:
    Input: [color_id, 90 object features]
    Old structure: [color_id, category_id, 61 object features]
    
    Changes:
    1. Removed category_embedding (not in new dataset)
    2. Updated object_processor input from 61 to 90 dimensions
    3. Updated output_dim calculation (removed category_emb_dim)
    """
    def __init__(self, num_colors, num_objects, 
                 color_emb_dim=16, object_feature_dim=128):
        super().__init__()
        
        # Color embedding (unchanged)
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        
        # REMOVED: category_embedding (not in new dataset)
        
        # Object processor - UPDATED input dimension from 61 to 90
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),  # Changed from 61 to num_objects (90)
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )

        # UPDATED output dimension: color_emb + object_feature (no category)
        self.output_dim = color_emb_dim + object_feature_dim
        print(f"MetadataEncoder output dimension: {self.output_dim} (color:{color_emb_dim} + objects:{object_feature_dim})")

    def forward(self, metadata):
        """
        Args:
            metadata: [batch, 91] where:
                - metadata[:, 0] = color_id (scalar)
                - metadata[:, 1:] = 90 object features (multi-hot)
        
        OLD structure was: [batch, 63] = [color, category, 61 objects]
        NEW structure is:  [batch, 91] = [color, 90 objects]
        """
        # Extract color ID (first element)
        color_ids = metadata[:, 0].long()
        
        # REMOVED: category_ids extraction (not in new data)
        
        # Extract object features (remaining 90 elements)
        object_features_raw = metadata[:, 1:]  # Changed from [:, 2:] to [:, 1:]
        object_features_raw = object_features_raw.float()

        # Get color embedding
        color_vec = self.color_embedding(color_ids)
        
        # REMOVED: category_vec calculation
        
        # Process objects
        object_vec = self.object_processor(object_features_raw)

        # Concatenate: color + objects (no category)
        combined_features = torch.cat([color_vec, object_vec], dim=1)
        
        return combined_features

# --- Decoder (Unchanged from modified version with global context) ---
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        self.rnn_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        print(f"Decoder RNN input dimension: {self.rnn_input_dim}")
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)

        rnn_input = torch.cat((
            embedded,
            context_permuted,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context.squeeze(1)


In [18]:
# ==================================================================================
# MODIFIED SEQ2SEQ CLASS
# WHAT CHANGED:
# 1. Renamed 'teacher_forcing_ratio' to 'text_teacher_forcing_ratio' for clarity.
# 2. Added a new parameter: 'meta_teacher_forcing_ratio' (default=1.0).
# 3. Added new logic inside forward():
#    - With a probability of 'meta_teacher_forcing_ratio', we use the GROUND TRUTH
#      metadata (the old, "easy" way).
#    - With a probability of (1 - meta_teacher_forcing_ratio), we use the
#      metadata PREDICTED by the meta_head (the new, "hard" way).
#
# WHY:
# This change fixes the "Metadata Exposure Bias." It couples the meta_head's
# performance to the text_loss. If the meta_head predicts garbage, the decoder
# is now forced to use it, which will cause the text_loss to be high. This
# sends a gradient signal back, forcing the meta_head to learn.
# ==================================================================================
class Seq2Seq(nn.Module):
    """
    REFACTORED to handle new metadata structure and remove category prediction.
    """
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_emb_dim=16, object_feature_dim=128, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        
        # UPDATED: MetadataEncoder no longer takes num_categories or category_emb_dim
        self.meta_encoder = MetadataEncoder(
            num_colors, 
            num_objects,
            color_emb_dim, 
            object_feature_dim
        )

        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                                 meta_features_dim, dec_layers, pad_id, dropout)

        # UPDATED: Metadata prediction head now outputs only color + objects (no category)
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects) # Removed num_categories
        )
        self.num_colors = num_colors
        # REMOVED: self.num_categories
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, 
                text_teacher_forcing_ratio=0.5, meta_teacher_forcing_ratio=1.0):
        """
        UPDATED forward pass - now supports scheduled metadata sampling
        
        Returns:
            text_logits: [batch, target_len-1, vocab_size]
            pred_color: [batch, num_colors]
            pred_object: [batch, num_objects]
        """
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        # Get global EEG context
        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        # Predict metadata (from EEG)
        meta_preds = self.meta_head(global_eeg_context)
        pred_color = meta_preds[:, :self.num_colors]
        pred_object = meta_preds[:, self.num_colors:] # Changed from slicing after categories

        # --- START OF NEW SCHEDULED SAMPLING LOGIC ---

        # Decide whether to use true metadata (teacher forcing) or predicted metadata
        use_true_meta = random.random() < meta_teacher_forcing_ratio
        
        if use_true_meta:
            # Use the ground-truth metadata (what we did before)
            meta_features = self.meta_encoder(metadata)
        else:
            # --- This is the new, "hard" path ---
            # Use the metadata *predicted* from the EEG.
            # We must build the metadata vector in the same [color_id, 90_objects]
            # format that the meta_encoder expects.
            
            with torch.no_grad(): # Don't prop gradients back *from* this creation step
                # Get predicted color ID (use argmax, not logits)
                pred_color_id_vec = pred_color.argmax(dim=-1).float().unsqueeze(1)
                
                # Get predicted object vector (use sigmoid > 0.5, not logits)
                pred_object_vec = (torch.sigmoid(pred_object) > 0.5).float()

                # Combine them into the [batch, 91] vector
                predicted_meta_vector = torch.cat([
                    pred_color_id_vec,
                    pred_object_vec
                ], dim=1)
            
            # Now, encode this *predicted* vector
            # We *do* want gradients to flow through this step
            meta_features = self.meta_encoder(predicted_meta_vector)

        # --- END OF NEW LOGIC ---

        # Text generation loop (unchanged)
        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                meta_features,  # <-- This is the key (now either true or predicted)
                global_eeg_context
            )

            outputs[t] = output
            teacher_force = random.random() < text_teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        # UPDATED: Return only 3 values (removed pred_category)
        return outputs[1:].permute(1, 0, 2), pred_color, pred_object

In [21]:
# ==================================================================================
# TRAINING AND EVALUATION - MAJOR CHANGES
# ==================================================================================
# ==================================================================================
# MODIFIED train_one_epoch FUNCTION
# WHAT CHANGED:
# 1. Added 'color_loss_weight' and 'object_loss_weight' to the signature.
# 2. Added 'meta_teacher_forcing_ratio' to the signature.
# 3. Passed 'meta_teacher_forcing_ratio' to the model() call.
# 4. Updated the final 'loss' calculation to use the new weights.
# 5. Added individual loss trackers (total_loss_t, _c, _o) and returned them.
#
# WHY:
# To pass the new scheduled sampling ratio to the model, apply the heavier
# metadata loss weights, and log the individual losses to see if the
# meta_head is actually learning.
#
# WHAT CHANGED (REVERTED):
# 1. Changed 'loss_o = object_criterion(torch.sigmoid(pred_object), ...)'
#    BACK TO 'loss_o = object_criterion(pred_object, ...)'
#
# WHY:
# We are now using nn.BCEWithLogitsLoss (the correct fallback), which
# expects raw LOGITS, not sigmoid probabilities. Applying sigmoid
# manually would be incorrect.
# ==================================================================================
def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion, object_criterion,
                    granger_edge_index, granger_edge_attr, 
                    color_loss_weight, object_loss_weight, 
                    meta_teacher_forcing_ratio=1.0):
    """
    UPDATED: Now accepts color/object weights and meta_teacher_forcing_ratio
    """
    model.train()
    total_loss = 0.0
    total_loss_t, total_loss_c, total_loss_o = 0.0, 0.0, 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        optimizer.zero_grad()

        # UPDATED: Pass the new ratio to the model
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, 
            granger_edge_index, granger_edge_attr, 
            text_teacher_forcing_ratio=0.5, # This is the text TF, keep it at 0.5
            meta_teacher_forcing_ratio=meta_teacher_forcing_ratio # This is the new one
        )

        # Calculate losses
        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        
        # --- THIS LINE IS THE FIX (REVERTED) ---
        # We pass raw logits to BCEWithLogitsLoss
        loss_o = object_criterion(pred_object, meta_b[:, 1:].float())

        # UPDATED: Combined loss with new weights
        loss = loss_t + (color_loss_weight * loss_c) + (object_loss_weight * loss_o)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_loss_t += loss_t.item()
        total_loss_c += loss_c.item()
        total_loss_o += loss_o.item()

        progress_bar.set_postfix(
            loss=loss.item(), 
            txt=loss_t.item(), 
            clr=loss_c.item(), 
            obj=loss_o.item()
        )

    n = len(loader)
    return total_loss / n, total_loss_t / n, total_loss_c / n, total_loss_o / n

@torch.no_grad()
# ==================================================================================
# MODIFIED evaluate FUNCTION
# WHAT CHANGED:
# 1. Added 'color_loss_weight' and updated 'object_loss_weight' in the signature.
# 2. Updated the final 'loss' calculation to use the new weights.
# 3. Added individual loss trackers (total_loss_t, _c, _o) and returned them.
# 4. Explicitly set 'text_teacher_forcing_ratio=0.0' in the model() call.
#
# WHY:
# To ensure the validation loss is calculated the same way as the training
# loss (using the same weights) for a fair comparison.
# We set text_teacher_forcing_ratio=0.0 (no teacher forcing) and do *not* pass
# the meta_teacher_forcing_ratio (letting it default to 1.0) to get a stable,
# consistent evaluation of the model's "best case" performance (i.e., when
# given true metadata). Changed it to 0 now.
#
# WHAT CHANGED (REVERTED):
# 1. Changed 'loss_o = object_criterion(torch.sigmoid(pred_object), ...)'
#    BACK TO 'loss_o = object_criterion(pred_object, ...)'
#
# WHY:
# We are now using nn.BCEWithLogitsLoss (the correct fallback), which
# expects raw LOGITS, not sigmoid probabilities.
# ==================================================================================
@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
             granger_edge_index, granger_edge_attr, 
             color_loss_weight, object_loss_weight):
    """
    UPDATED: Removed category_criterion, added loss weights, return all losses
    """
    model.eval()
    total_loss = 0.0
    total_loss_t, total_loss_c, total_loss_o = 0.0, 0.0, 0.0
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        # UPDATED: Model now returns only 3 values
        # We explicitly set text_teacher_forcing_ratio=0.0 for evaluation
        # We use the default meta_teacher_forcing_ratio=1.0 (true metadata)
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, 
            granger_edge_index, granger_edge_attr, 
            text_teacher_forcing_ratio=0.0, # <-- No text teacher forcing
            meta_teacher_forcing_ratio=0.0  # <-- Always use true metadata
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        # REMOVED: loss_cat calculation
        
        # --- THIS LINE IS THE FIX (REVERTED) ---
        # We pass raw logits to BCEWithLogitsLoss
        loss_o = object_criterion(pred_object, meta_b[:, 1:].float())

        # UPDATED: Combined loss with new weights
        loss = loss_t + (color_loss_weight * loss_c) + (object_loss_weight * loss_o)
        
        total_loss += loss.item()
        total_loss_t += loss_t.item()
        total_loss_c += loss_c.item()
        total_loss_o += loss_o.item()
        
        progress_bar.set_postfix(
            loss=loss.item(), 
            txt=loss_t.item(), 
            clr=loss_c.item(), 
            obj=loss_o.item()
        )

    n = len(loader)
    return total_loss / n, total_loss_t / n, total_loss_c / n, total_loss_o / n

In [22]:
# ==================================================================================
# MAIN EXECUTION
# ==================================================================================

if __name__ == "__main__":
    # Create dataset and loaders
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(dataset)
    n_train = int(N * TRAIN_PCT)
    n_val = int(N * VAL_PCT)
    n_test = N - n_train - n_val
    g = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

    # Create Granger matrix
    print("Creating Granger Causality matrix...")
    try:
        eeg_b, _, _ = next(iter(train_loader))
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        num_channels = eeg_b.shape[1]

        granger_edge_index, granger_edge_attr = add_self_loops(
            granger_edge_index,
            edge_attr=granger_edge_attr,
            num_nodes=num_channels,
            fill_value=1.0
        )

        if granger_edge_attr is None:
            granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float)

        granger_edge_index = granger_edge_index.to(torch.long).to(device)
        granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
        print(f"Granger matrix created: {granger_edge_index.shape}")

    except Exception as e:
        print(f"Error creating Granger matrix: {e}. Using fallback.")
        num_channels = 62
        edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
        granger_edge_index = edge_index.to(torch.long).to(device)
        granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float32).to(device)

    # UPDATED: Instantiate model with new parameters (no num_categories)
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,     # 12 instead of 77
        num_objects=NUM_OBJECTS,   # 90 instead of 61
        pad_id=PAD_ID,
        dropout=0.2,
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2
    ).to(device)

    print(f"Model instantiated on '{device}'.")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # UPDATED: Setup losses (removed category_criterion)
    object_criterion = focal_loss_criterion
    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    color_criterion = nn.CrossEntropyLoss()
    # REMOVED: category_criterion

    optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)

    # ==============================================================================
    # --- UPDATED TRAINING LOOP (NEW SCHEDULE AND LOGGING) ---
    # ==============================================================================
    EPOCHS = 40  # <-- Increased epochs
    best_val_loss = float('inf')
    
    # --- Set high loss weights for metadata tasks ---
    OBJECT_LOSS_WEIGHT = 10.0
    COLOR_LOSS_WEIGHT = 1.0

    # --- NEW: Metadata Teacher Forcing Schedule ---
    # This schedule anneals from 1.0 (100% true meta) down to 0.5 (50% true meta)
    # over the first 20 epochs.
    meta_tf_schedule = np.linspace(1.0, 0.5, 20)

    print("\n--- Starting Training ---")
    print(f"Object Loss Weight: {OBJECT_LOSS_WEIGHT} | Color Loss Weight: {COLOR_LOSS_WEIGHT}")

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()

        # Get the ratio for this epoch (clip at the end)
        if epoch - 1 < len(meta_tf_schedule):
            current_meta_tf_ratio = meta_tf_schedule[epoch - 1]
        else:
            current_meta_tf_ratio = meta_tf_schedule[-1] # Stay at 0.5
            
        print(f"\nUsing Metadata Teacher Forcing Ratio: {current_meta_tf_ratio:.2f}")

        # UPDATED: Pass weights and new ratio to train_one_epoch
        train_loss, tr_t, tr_c, tr_o = train_one_epoch(
            model, train_loader, optimizer,
            text_criterion, color_criterion, object_criterion,
            granger_edge_index, granger_edge_attr, 
            COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT,
            meta_teacher_forcing_ratio=current_meta_tf_ratio
        )
        
        # UPDATED: Pass weights to evaluate
        val_loss, val_t, val_c, val_o = evaluate(
            model, val_loader,
            text_criterion, color_criterion, object_criterion,
            granger_edge_index, granger_edge_attr, 
            COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT
        )

        scheduler.step(val_loss)
        end_time = time.time()
        epoch_mins = int((end_time - start_time) / 60)
        epoch_secs = int((end_time - start_time) % 60)

        # UPDATED: Print all individual losses
        print(f'\nEpoch: {epoch:02} | Time: {epoch_mins}m {epoch_secs}s')
        print(f'\tTrain Loss: {train_loss:.4f} | Txt: {tr_t:.4f} | Clr: {tr_c:.4f} | Obj: {tr_o:.4f}')
        print(f'\t  Val Loss: {val_loss:.4f} | Txt: {val_t:.4f} | Clr: {val_c:.4f} | Obj: {val_o:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model_save_path = 'eeg-meta-text-qwen-refactored-model.pt'
            torch.save(model.state_dict(), model_save_path)
            print(f"\t-> Val loss decreased. Saving best model to '{model_save_path}'")
        else:
            print("\t-> Val loss did not improve.")

    print("\n--- Training Complete ---")

    # ==============================================================================
    # --- INFERENCE SECTION (NO CHANGES NEEDED HERE) ---
    # ==============================================================================
    
    # Load best model
    checkpoint_path = 'eeg-meta-text-qwen-refactored-model.pt'
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"\nBest model '{checkpoint_path}' loaded for inference.")

    # Load object mapping (UPDATED path to new mapping file)
    OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_qwen.json"
    try:
        with open(OBJECT_MAPPING_FILE, 'r') as f:
            object_mapping = json.load(f)
        print(f"Object mapping loaded: {len(object_mapping)} objects")
    except FileNotFoundError:
        print(f"Warning: '{OBJECT_MAPPING_FILE}' not found.")
        object_mapping = {}

    @torch.no_grad()
    def generate_end_to_end(model, eeg_signal, edge_index, edge_attr,
                            sample_idx,
                            k=5,
                            penalty_alpha=0.3,
                            context_beta=0.7,
                            max_len=100):
        """
        UPDATED: Modified to handle new metadata structure (no category prediction)
        
        Returns:
            predicted_text: str
            pred_color: float (color ID)
            pred_object_ids: list of int (object IDs with prob > 0.5)
        """
        model.eval()
        eeg_signal = eeg_signal.unsqueeze(0).to(device)

        # 1. Encode EEG
        encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
        
        # 2. Get global EEG context
        hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        # 3. Predict metadata from EEG
        meta_preds_logits = model.meta_head(global_eeg_context)

        # UPDATED: Extract only color and objects (no category)
        pred_color_logits = meta_preds_logits[:, :model.num_colors]
        pred_object_logits = meta_preds_logits[:, model.num_colors:] # Changed indexing

        # For decoder input
        pred_color_id_vec = pred_color_logits.argmax(dim=-1).float().unsqueeze(1)
        pred_object_vec = (torch.sigmoid(pred_object_logits) > 0.5).float()

        # UPDATED: Create metadata vector with new structure [color, 90 objects]
        predicted_meta_vector = torch.cat([
            pred_color_id_vec,
            pred_object_vec
        ], dim=1)
        
        # For printing
        pred_color_for_print = pred_color_id_vec.item()
        # REMOVED: pred_category_for_print
        
        # Get all predicted object IDs
        object_probs = torch.sigmoid(pred_object_logits)
        pred_object_ids_list = (object_probs > 0.5).nonzero(as_tuple=True)[1].tolist()
        
        # 5. Encode the predicted metadata
        predicted_meta_features = model.meta_encoder(predicted_meta_vector)

        # 6. Initialize decoder
        decoder_hidden = model.decoder.init_hidden(encoder_hidden)

        if sample_idx < 2:
            print(f"\n--- [Sample {sample_idx+1}] Generation Start (End-to-End) ---")

        # 7. Generation loop
        generated_ids = torch.tensor([SOS_ID], device=device)
        for step in range(max_len):
            input_token = generated_ids[-1].unsqueeze(0)

            prediction, new_hidden, attention_context = model.decoder(
                input_token,
                decoder_hidden,
                encoder_outputs,
                predicted_meta_features,
                global_eeg_context
            )
            decoder_hidden = new_hidden
            
            model_log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
            topk_model_log_probs, topk_ids = torch.topk(model_log_probs, k)
            
            current_seq_len = generated_ids.shape[0]
            prev_token_embeddings = F.normalize(model.decoder.embedding(generated_ids), dim=-1)
            candidate_token_embeddings = F.normalize(model.decoder.embedding(topk_ids), dim=-1)
            
            sim_matrix = torch.matmul(candidate_token_embeddings, prev_token_embeddings.t())
            degeneration_penalty = torch.zeros(k, device=device)
            if current_seq_len > 1:
                degeneration_penalty, _ = torch.max(sim_matrix, dim=-1)
                
            current_decoder_state = F.normalize(decoder_hidden[-1].squeeze(), dim=-1)
            context_agreement_score = torch.matmul(candidate_token_embeddings, current_decoder_state)
            
            final_score = topk_model_log_probs + context_beta * context_agreement_score - penalty_alpha * degeneration_penalty
            
            best_next_token_idx = torch.argmax(final_score)
            next_token_id = topk_ids[best_next_token_idx]

            generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)])
            if next_token_id.item() == EOS_ID:
                if sample_idx < 5:
                    print("  [EOS Reached]")
                break
                
        if generated_ids.numel() > 1:
            predicted_text_ids = generated_ids[1:-1] if generated_ids[-1].item() == EOS_ID else generated_ids[1:]
            predicted_text = tokenizer.decode(predicted_text_ids.tolist(), skip_special_tokens=True)
        else:
            predicted_text = ""
        
        if sample_idx < 5:
            print(f"--- [Sample {sample_idx+1}] Generation End ---")

        # UPDATED: Return only 3 values (no category)
        return predicted_text, pred_color_for_print, pred_object_ids_list

    # Run inference
    NUM_SAMPLES = 20
    print(f"\n--- Running TRUE END-TO-END Inference on {NUM_SAMPLES} Samples ---")

    predictions = []
    references = []

    for i in range(NUM_SAMPLES):
        eeg_sample, meta_sample, true_text_ids = test_ds[i]

        # Extract true metadata (UPDATED for new structure)
        true_color_id = int(meta_sample[0].item()) # First element is color
        # REMOVED: true_category_id extraction
        true_object_vector = meta_sample[1:] # Remaining 90 elements are objects
        true_object_ids_tensors = true_object_vector.nonzero(as_tuple=True)[0]
        true_object_names = [object_mapping.get(str(id_item), f"ID:{id_item}") 
                             for id_item in true_object_ids_tensors.tolist()]
        if not true_object_names:
            true_object_names = ["None"]

        # UPDATED: Generate with new function signature (no category)
        predicted_text, pred_color, pred_object_ids = generate_end_to_end(
            model,
            eeg_sample,
            granger_edge_index,
            granger_edge_attr,
            sample_idx=i,
            k=5,
            penalty_alpha=0.3,
            context_beta=0.7
        )

        # Decode true text
        true_text_ids_list = true_text_ids.long().tolist()
        true_text = tokenizer.decode(true_text_ids_list, skip_special_tokens=True)

        predictions.append(predicted_text)
        references.append(true_text)

        # Convert predicted object IDs to names
        pred_object_names = [object_mapping.get(str(oid), f"ID:{oid}") 
                             for oid in pred_object_ids]
        if not pred_object_names:
            pred_object_names = ["None"]

        # UPDATED: Print summary without category
        print(f"\n--- Sample {i+1}/{NUM_SAMPLES} Summary (Index: {i}) ---")
        print(f"GROUND TRUTH TEXT: {true_text}")
        print(f"MODEL PREDICTION TEXT: {predicted_text}")
        print("\nMETADATA PREDICTION (FROM EEG):")
        print(f"  Color ID:      Truth={true_color_id}, Predicted={pred_color}")
        # REMOVED: Category ID line
        print(f"  Object(s):     Truth={', '.join(true_object_names)}, Predicted={', '.join(pred_object_names)}")
        print("-" * 50)

    # Evaluation metrics
    print("\n--- Evaluation Metrics (True End-to-End Model) ---")
    try:
        # --- FIXED: Use hf_evaluate alias ---
        bleu_metric = hf_evaluate.load('bleu')
        bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
        print(f"BLEU Score: {bleu_results['bleu']:.4f}")

        rouge_metric = hf_evaluate.load('rouge')
        rouge_results = rouge_metric.compute(predictions=predictions, references=references)
        print(f"ROUGE-1 Score: {rouge_results['rouge1']:.4f}")
        print(f"ROUGE-2 Score: {rouge_results['rouge2']:.4f}")
        print(f"ROUGE-L Score: {rouge_results['rougeL']:.4f}")
    except Exception as e:
        print(f"Could not calculate evaluation metrics: {e}")

Creating Granger Causality matrix...
Granger matrix created: torch.Size([2, 2457])
Encoder RNN input size: 256
MetadataEncoder output dimension: 144 (color:16 + objects:128)
Decoder RNN input dimension: 1424
Model instantiated on 'cuda'.
Total parameters: 19,875,552

--- Starting Training ---
Object Loss Weight: 10.0 | Color Loss Weight: 1.0

Using Metadata Teacher Forcing Ratio: 1.00


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 01 | Time: 7m 38s
	Train Loss: 9.4456 | Txt: 5.7063 | Clr: 2.2824 | Obj: 0.1457
	  Val Loss: 8.1294 | Txt: 4.9551 | Clr: 2.2183 | Obj: 0.0956
	-> Val loss decreased. Saving best model to 'eeg-meta-text-qwen-refactored-model.pt'

Using Metadata Teacher Forcing Ratio: 0.97


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

...

Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 06 | Time: 7m 36s
	Train Loss: 7.0825 | Txt: 3.8922 | Clr: 2.2293 | Obj: 0.0961
	  Val Loss: 7.6627 | Txt: 4.5070 | Clr: 2.2131 | Obj: 0.0943
	-> Val loss did not improve.

Using Metadata Teacher Forcing Ratio: 0.84


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [13]:
import evaluate as hf_evaluate
print("\n--- Evaluation Metrics (True End-to-End Model) ---")
try:
    # --- USE THE ALIAS 'hf_evaluate' ---
    bleu_metric = hf_evaluate.load('bleu')
    bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    print(f"BLEU Score: {bleu_results['bleu']:.4f}")

    # --- USE THE ALIAS 'hf_evaluate' ---
    rouge_metric = hf_evaluate.load('rouge')
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    print(f"ROUGE-1 Score: {rouge_results['rouge1']:.4f}")
    print(f"ROUGE-2 Score: {rouge_results['rouge2']:.4f}")
    print(f"ROUGE-L Score: {rouge_results['rougeL']:.4f}")
    
except Exception as e:
    print(f"Could not calculate evaluation metrics: {e}")


--- Evaluation Metrics (True End-to-End Model) ---
BLEU Score: 0.0236
ROUGE-1 Score: 0.1966
ROUGE-2 Score: 0.0208
ROUGE-L Score: 0.1901
